# Notebook 2 · Real EEG data: shapes, masks, and signals

The UCI **EEG Eye State** dataset contains 14 EEG channels from one continuous
117-second measurement, with eye state labelled from video (`0` open, `1` closed).

It contains one participant and is not an `epochs × channels × time` dataset.

Roesler, O. (2013), UCI Machine Learning Repository, CC BY 4.0,
<https://doi.org/10.24432/C57G7J>.

In [ ]:
from pathlib import Path

class Check:
    def _result(self, passed, success, hint):
        if passed:
            print(f"✅ {success}")
        else:
            print("✗ Not correct. Open the hint if needed.")
        return passed

    def equal(self, actual, expected, success="Correct.", hint="Value does not match the expected result."):
        try:
            passed = actual == expected
            if hasattr(passed, "all"):
                passed = bool(passed.all())
        except Exception:
            passed = False
        return self._result(bool(passed), success, hint)

    def shape(self, actual, expected, success="Shape is correct.", hint="Shape is incorrect."):
        return self._result(tuple(actual.shape) == tuple(expected), success, hint)

    def columns(self, frame, expected, success="Columns are correct.", hint="Inspect frame.columns and select with a list of names."):
        return self._result(list(frame.columns) == list(expected), success, hint)

    def choice(self, actual, expected, explanations):
        normalised = str(actual).strip().upper()
        hint = explanations.get(normalised, "Choose one of the listed letters.")
        return self._result(normalised == expected.upper(), "Correct.", hint)

check = Check()

## 1 · Load the ARFF file

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import arff

candidates = [
    Path("book/data/real/eeg_eye_state.arff"),
    Path("../data/real/eeg_eye_state.arff"),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Open this notebook from the workshop repository.")

raw_records, metadata = ...
eeg = ...
eeg["eyeDetection"] = ...
eeg.head()

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Use `arff.loadarff(data_path)`, pass the returned records to `pd.DataFrame`, and convert `eyeDetection` with `.astype(int)`.

</details>

In [ ]:
check.shape(eeg, (14980, 15), hint="The completed DataFrame should have 14 channels and one label column.")
check.equal(str(eeg["eyeDetection"].dtype).startswith("int"), True, hint="Convert eyeDetection to integers.")

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
raw_records, metadata = arff.loadarff(data_path)
eeg = pd.DataFrame(raw_records)
eeg["eyeDetection"] = eeg["eyeDetection"].astype(int)
eeg.head()
```

</details>

## 2 · Predict the shape

The dataset has 14 EEG channels and one label column. What is the shape of the complete
DataFrame?

- A: `(14980, 14)`
- B: `(14980, 15)`
- C: `(15, 14980)`

In [ ]:
answer_shape = ""

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Count the 14 EEG channels and the `eyeDetection` label column.

</details>

In [ ]:
check.choice(
    answer_shape,
    "B",
    {
        "A": "That counts only features and forgets the target column.",
        "B": "",
        "C": "pandas uses rows × columns, not columns × rows.",
    },
)

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

**B: `(14980, 15)`.** The 14 channel columns plus `eyeDetection` give 15 columns.
The 14,980 measurements are rows.

</details>

## 3 · Separate features and target

Fill the blanks so `X` contains the channels and `y` contains eye state.

In [ ]:
X = eeg.drop(columns=[...])
y = eeg[...]

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Remove `eyeDetection` from `X` and select that same column as `y`.

</details>

In [ ]:
check.shape(X, (14980, 14), hint="Drop the eyeDetection label from the feature table.")
check.equal(y.name, "eyeDetection", hint="Select the label as a Series with one column name.")

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
X = eeg.drop(columns=["eyeDetection"])
y = eeg["eyeDetection"]
```

</details>

### Reflection · A target hiding among the features

Imagine leaving `eyeDetection` inside `X` before training a classifier to predict
`y`. What would happen to performance, and why would a very high score be a warning
rather than a success?

In [ ]:
reflection_leakage = """
Performance would ... because ...
"""

<details class="notebook-reflection">
<summary><strong>Compare your reasoning</strong></summary>

The model would receive the answer as an input feature, so performance could become
nearly perfect without learning anything about EEG. This is target leakage. Suspiciously
strong results should prompt an inspection of feature names, preprocessing order, and
whether information from the target or test set entered the features.

</details>

## 4 · Move from pandas to NumPy

Create `signals` as a NumPy array. Then select the first 100 samples from channel O1.
The output should be one-dimensional.

In [ ]:
signals = ...
o1_index = list(X.columns).index("O1")
o1_excerpt = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Use `.to_numpy()` for `signals`. Select rows `:100` and column `o1_index`.

</details>

In [ ]:
check.shape(signals, (14980, 14))
check.shape(o1_excerpt, (100,), hint="Select rows 0:100 and one channel column.")

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
signals = X.to_numpy()
o1_index = list(X.columns).index("O1")
o1_excerpt = signals[:100, o1_index]
```

</details>

## 5 · Boolean masks

Use `y` to make two arrays: samples recorded with eyes open and samples recorded with
eyes closed.

In [ ]:
eyes_open = ...
eyes_closed = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

`y.eq(0)` and `y.eq(1)` create Boolean row masks.

</details>

In [ ]:
check.equal(eyes_open.shape[1], 14, hint="Use a Boolean row mask; retain every channel.")
check.equal(eyes_closed.shape[1], 14, hint="Use a Boolean row mask; retain every channel.")
check.equal(len(eyes_open) + len(eyes_closed), len(signals), hint="Every sample should belong to exactly one state.")

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
eyes_open = signals[y.eq(0)]
eyes_closed = signals[y.eq(1)]
```

</details>

## 6 · Aggregate along the correct axis

Calculate one mean value per channel for each eye state. The result should have shape
`(14,)`.

In [ ]:
open_channel_means = ...
closed_channel_means = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Rows contain measurements and columns contain channels. Average the row axis.

</details>

In [ ]:
check.shape(open_channel_means, (14,), hint="Rows are samples. Which axis should disappear?")
check.shape(closed_channel_means, (14,), hint="Rows are samples. Which axis should disappear?")

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
open_channel_means = eyes_open.mean(axis=0)
closed_channel_means = eyes_closed.mean(axis=0)
```

Axis 0 contains measurements. Averaging it leaves one mean per channel.

</details>

### Reflection · What did the mean remove?

After averaging all samples within each eye state, which kinds of information are gone?
Could two recordings have identical channel means but meaningfully different signals?
Give one example.

In [ ]:
reflection_averaging = """
The mean removes ...
Two recordings could differ in ...
"""

<details class="notebook-reflection">
<summary><strong>Compare your reasoning</strong></summary>

The means remove temporal order, transitions, variability, oscillatory structure, and
the distribution of values. Two recordings could share the same mean while differing
in variance, spectral power, artefacts, or the timing of state changes. Aggregation is
useful only when the retained summary matches the scientific question.

</details>

## 7 · Visual comparison

Make a grouped or paired plot comparing the 14 channel means. Label the axes and states.
Then answer: why would this plot alone be insufficient evidence that closing the eyes
*caused* the observed differences?

In [ ]:
# Create a grouped bar chart with one pair of bars per channel.


# Why this does not establish causation:

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

Use `np.arange` for channel positions and offset the open/closed bars by half a bar width. Consider the number of participants, chronological dependence, artefacts, and experimental control.

</details>

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
channel_names = X.columns
positions = np.arange(len(channel_names))
width = 0.4

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(positions - width / 2, open_channel_means, width, label="Eyes open")
ax.bar(positions + width / 2, closed_channel_means, width, label="Eyes closed")
ax.set_xticks(positions, channel_names, rotation=45)
ax.set(xlabel="Channel", ylabel="Mean EEG value")
ax.legend()
plt.show()
```

The recording contains one participant and a chronological sequence rather than
independent, randomly assigned observations. Artefacts, drift, time, and transitions
between states could contribute to the difference. The plot is descriptive.

</details>

## Bonus · Build pseudo-epochs

Take the first 14,000 samples and reshape them into
`100 pseudo-epochs × 140 time samples × 14 channels`, then transpose to the ACN
convention `epochs × channels × time`.

These fixed-width chunks are not experimentally defined epochs.

In [ ]:
pseudo_epochs = ...

<details class="notebook-hint">
<summary><strong>Hint</strong></summary>

First reshape the first 14,000 rows to `(100, 140, 14)`, then transpose axes 1 and 2.

</details>

In [ ]:
check.shape(pseudo_epochs, (100, 14, 140), hint="First reshape to (100, 140, 14), then transpose the final two axes.")

<details class="notebook-answer">
<summary><strong>Reveal answer</strong></summary>

```python
pseudo_epochs = (
    signals[:14000]
    .reshape(100, 140, 14)
    .transpose(0, 2, 1)
)
```

The final shape is `pseudo-epochs × channels × time`: `(100, 14, 140)`.

</details>

### Reflection · A valid shape is not yet a valid analysis

The pseudo-epochs have the expected three-dimensional shape. Why does that not make
them genuine experimental epochs? What event information would real epoching require?

In [ ]:
reflection_epoching = """
These chunks are not genuine epochs because ...
Real epoching would require ...
"""

<details class="notebook-reflection">
<summary><strong>Compare your reasoning</strong></summary>

Reshaping creates equal-width chunks but does not align them to experimental events.
Real epoching needs event markers, event identities, a sampling rate, a defined time
window, and decisions about baselines, artefacts, and boundary cases. Correct dimensions
cannot replace experimental meaning.

</details>